# 05 · Fases del ERP y unión con el MES

En este notebook se unen las dos ramas del proyecto: las fases del ERP, que aportan el tiempo teórico, y las fases
del MES (notebook 03), que aportan el tiempo real. Antes de unirlas se limpian las fases del ERP y se conservan solo
las de las OF válidas (notebook 04). Después se descartan las fases sin tiempo teórico y los tipos de fase que no
forman parte de la población de estudio.

- **Entrada:** `data/raw/CPRMOTask.csv`, `data/raw/maping_fase.csv`, `data/processed/df_OF.parquet` y `data/processed/df_fases_MES.parquet`
- **Salida:** `data/processed/df_FASE_final.parquet` y `results/tables/embudo_05.json`

In [61]:
import sys
from pathlib import Path

# Raíz del repositorio (la carpeta que contiene src/), para poder importar src
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd

from src.columnas import COLUMNAS_FASES, COLUMNAS_MAPEO, leer_csv
from src.embudo import guardar_embudo, tabla_embudo
from src.limpieza import eliminar_columnas
from src.rutas import PROCESSED

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

embudo = {}  # número de fases que quedan tras cada paso

## 1. Carga de las fases del ERP

In [62]:
df_FASE = leer_csv("CPRMOTask.csv", COLUMNAS_FASES)
embudo["Fases iniciales"] = len(df_FASE)
print(df_FASE.shape)

(116055, 86)


## 2. Fases externas o canceladas

Las fases externas (subcontratadas) o canceladas no sirven para entrenar el modelo.

In [63]:
def filtrar_fases_no_validas(df):
    """Conserva solo las fases que no son externas ni canceladas.
    Descarta cualquier fase con un 1 en External o en Canceled (un NULL cuenta como 0).
    Devuelve un DataFrame nuevo (no modifica el original)."""
    n_inicial = len(df)
    mask_validas = (df["External"].fillna(0) == 0) & (df["Canceled"].fillna(0) == 0)
    df_out = df[mask_validas].copy()

    # Desglose informativo por motivo (pueden solaparse)
    print(f"Partida: {n_inicial} fases")
    print(f"  Externas:   {(df['External'].fillna(0) == 1).sum()}")
    print(f"  Canceladas: {(df['Canceled'].fillna(0) == 1).sum()}")
    print(f"Conservadas: {len(df_out)} fases (descartadas {n_inicial - len(df_out)})")
    return df_out


df_FASE = filtrar_fases_no_validas(df_FASE)

Partida: 116055 fases
  Externas:   8501
  Canceladas: 0
Conservadas: 107554 fases (descartadas 8501)


Puede haber fases externas que no estén marcadas como tales. Se buscan por el nombre de la fase.

In [64]:
posibles_externas = "ext|subcon"
mask_infiltradas = df_FASE["Description"].str.contains(posibles_externas, case=False, na=False)

print(f"Fases externas no marcadas como externas: {mask_infiltradas.sum()}")
print(df_FASE.loc[mask_infiltradas, "Description"].value_counts().to_string())

df_FASE = df_FASE[~mask_infiltradas]
embudo["Fases sin externas"] = len(df_FASE)
print(f"\nFases restantes: {len(df_FASE)}")

Fases externas no marcadas como externas: 12
Description
Fase Externa Mecanitzar Peça Acabada      5
Mecanització Externa Peça Centre          3
Fase Externa Mecanitzar Fase Posterior    2
Fase Externa Mecanitzar Fase Prèvia       2

Fases restantes: 107542


## 3. Limpieza de atributos

La tabla tiene 86 columnas. Antes de seleccionar los atributos, se revisan los que podrían ser constantes:

In [65]:
def inspeccionar_candidatos(df, columnas):
    """Muestra cuántos valores distintos tiene cada columna candidata y su distribución.
    Una columna constante (un solo valor) es prescindible. No elimina nada."""
    for col in columnas:
        print(f"====== {col} ======")
        print(df[col].value_counts(dropna=False).head(10).to_string(), "\n")


inspeccionar_candidatos(df_FASE, ["TimeUnit", "OccupancyHours", "OccupancyPercent"])

====== TimeUnit ======
TimeUnit
1    103958
2      3563
3        21 

====== OccupancyHours ======
OccupancyHours
24.0    107542 

====== OccupancyPercent ======
OccupancyPercent
100.0    107542 



`OccupancyHours` y `OccupancyPercent` son constantes y se pueden eliminar.

El único atributo con valores distintos es `TimeUnit`, que indica en qué unidad están los tiempos de preparación y
de ejecución de cada fase: 1 = minutos, 2 = horas, 3 = días, 4 = semanas, 5 = meses.

Hay 3.563 fases con `TimeUnit = 2` (horas) y 21 con `TimeUnit = 3` (días). Revisando las fases con `TimeUnit = 2`
con oficina técnica se ha comprobado que el error está en la unidad indicada: los valores numéricos de los tiempos sí
están en minutos y son correctos. Por eso no se convierte ningún tiempo y todos los tiempos teóricos se toman en
minutos. En el dataset final no queda ninguna fase con `TimeUnit = 3` (se comprueba en el notebook 08).

La columna `TimeUnit` se conserva solo para trazabilidad: no interviene en ningún cálculo y se elimina al construir el
dataset de entrenamiento (notebook 08).

In [66]:
COLUMNAS_SOBRAN = [
    "CodMOTask", "Notes", "Order", "RepetitionTime", "DelimitationType",
    "DelimitationDate", "AllowFraccionate", "MinimumFraction", "RepetitionLot", "CodCompany",
    "IDArticleExternal", "RepetitionTimeReal", "Group", "SeriesRequired", "QualityStatus",
    "InspectedDate", "Priority", "OptimizationFeature", "Fixed", "WithoutTimetable",
    "OccupancyType", "DelimitationObligatory", "IDBudgetCategory", "ResourcePlanType",
    "SplitNumber", "ManualPlanning", "IDBudgetEmployee", "IDBudgetMachine", "IDBudgetTool",
    "IDBudgetMaterial", "IDBudgetOtherCost", "RowTimestamp", "LastModifiedBy", "WorkUnits",
    "IDProductFormula", "LotsOrder", "PrintLabels", "AutoExecutionImputation", "External",
    "IDSiteMaterial", "IDWarehouseMaterial", "GroupingNumber", "CreationTimestamp",
    "RowCreatedBy", "BlockedMaterial", "BlockedCost", "ImputationCostUpdated", "IDMOTaskParent",
    "IDCheckList", "NotOverlapCode", "TiedCode", "GroupedManufacturing", "Blocked",
    "ReleaseKeptResources", "DelayMargin", "BlockedChangeMaterial", "IDHourType", "Color",
    "SetToQtyConversionFactor", "AdjustTaskToDay", "PercentProgressType", "StatusTaskMES",
    "OccupancyHours", "OccupancyPercent", "TaskType", "IDTask", "PercentProgress", "Canceled",
    "IDSupplier", "IDTaskType", "DateReplanify", "PlannedStartDate", "PlannedEndDate", "ManualEndDate",
]

df_FASE = eliminar_columnas(df_FASE, COLUMNAS_SOBRAN)
df_FASE.head(3)

Eliminadas 74 columnas. Quedan 12 columnas y 107542 filas.


,IDMOTask,IDManufacturingOrder,Description,PreparationTime,ExecutionTime,TimeUnit,ProductEntry,MaterialExit,IDUsualTask,Quantity,ManualStartDate,RealEndDate
0,00003219-f775-4bea-9e04-dbafb6e8fd80,7de89fb9-e8db-48bf-b822-9c52431c44a1,Marcatges,1,1,1,0,0,134bdf39-e99c-4be8-af7a-36b7899c152d,1.0,2026-01-19 15:58:00.000,2026-01-23 00:00:00.000
1,00016084-d562-4609-8501-3a761ca0351b,83c754de-86e0-4ec0-bf8b-c25d7a03fc3b,Marcatges,2,0,1,0,0,134bdf39-e99c-4be8-af7a-36b7899c152d,1.0,2025-06-24 15:58:00.000,NaN
2,0001b76c-daef-40a0-ab87-b634f1cf52ca,8a44da1e-e189-44f4-89c4-e4787480d008,Preparació de material,5,0,1,0,1,34abaa33-73f8-4ae5-aeef-ffed0d97b4b6,10.0,2025-08-18 14:03:00.000,NaN


## 4. Fases de OF válidas

Solo se conservan las fases de las OF válidas del notebook 04, es decir, de OF terminadas antes de agosto de 2026.
Así todas las fases tienen su tiempo imputado completo.

`IDManufacturingOrder` es la clave primaria de las OF y aparece en las fases como clave foránea: indica a qué OF
pertenece cada fase. Si una fase no pertenece a ninguna OF válida, se elimina.

In [67]:
df_OF = pd.read_parquet(PROCESSED / "df_OF.parquet")
print(f"OF válidas: {len(df_OF)}")

antes = len(df_FASE)
df_FASE = df_FASE[df_FASE["IDManufacturingOrder"].isin(df_OF["IDManufacturingOrder"])]
embudo["Fases en OF válidas"] = len(df_FASE)
print(f"Fases antes: {antes} | después: {len(df_FASE)}")

OF válidas: 11983
Fases antes: 107542 | después: 69986


Las fases eliminadas pertenecen a OF de las que no se puede garantizar que estén terminadas; por tanto, tampoco se
puede garantizar que sus tiempos reales estén completos.

## 5. Unión con las fases del MES

`df_FASE` viene del ERP y `df_fases_MES` del MES, así que los identificadores de las fases no coinciden entre ambas
tablas. En la base de datos existe una tabla de mapeo entre las entidades del ERP y las del MES, donde cada fila
tiene el identificador de cada sistema. Se añade a `df_FASE` el identificador del MES para poder relacionarla con
las imputaciones.

In [68]:
df_maping_fase = leer_csv("maping_fase.csv", COLUMNAS_MAPEO)
df_maping_fase = df_maping_fase[["ERPEntityId", "MESEntityId"]]

df_FASE = df_FASE.merge(df_maping_fase, left_on="IDMOTask", right_on="ERPEntityId", how="left")
print(f"Fases sin identificador del MES: {df_FASE['MESEntityId'].isna().sum()}")
df_FASE = df_FASE.drop(columns="ERPEntityId")  # duplica a IDMOTask

Fases sin identificador del MES: 0


In [69]:
df_fases_MES = pd.read_parquet(PROCESSED / "df_fases_MES.parquet")
print(f"Fases del MES: {len(df_fases_MES)}")

# En el MES los identificadores están en mayúsculas y en la tabla de mapeo en minúsculas
df_fases_MES["SmFaseUid"] = df_fases_MES["SmFaseUid"].str.lower().str.strip()
df_FASE["MESEntityId"] = df_FASE["MESEntityId"].str.lower().str.strip()

df_FASE_final = df_fases_MES.merge(df_FASE, left_on="SmFaseUid", right_on="MESEntityId", how="left")
print(f"Fases tras la unión: {len(df_FASE_final)}")
print(f"Fases del MES sin fase del ERP: {df_FASE_final['MESEntityId'].isna().sum()}")

Fases del MES: 17294
Fases tras la unión: 17294
Fases del MES sin fase del ERP: 1000


Tras cruzar las fases con tiempo real (MES) y su tiempo teórico correspondiente (ERP), se detectaron fases sin
correspondencia en el sistema teórico. Estas fases pertenecen a órdenes de fabricación que quedaron fuera del rango
temporal de finalización del estudio.

En la depuración previa, las órdenes de fabricación se filtraron por su fecha de finalización, ya que en una orden no
finalizada dentro del intervalo no puede garantizarse que sus imputaciones de tiempo estén completas, y el tiempo
real registrado podría no reflejar el trabajo total de la fase. Al descartar dichas órdenes del ERP, sus fases
quedaron sin tiempo teórico asociado.

Dado que estas fases carecen de tiempo teórico y que el ratio real/teórico constituye la variable objetivo, se
descartan del análisis. Esta exclusión es coherente con el filtrado temporal previamente adoptado y garantiza que el
conjunto final se componga únicamente de fases con imputaciones completas y ambos tiempos disponibles.

In [70]:
antes = len(df_FASE_final)
df_FASE_final = df_FASE_final[df_FASE_final["MESEntityId"].notna()]
embudo["Fases con imputaciones validadas"] = len(df_FASE_final)
print(f"Ha descartar imputaciones sin fase del ERP: {antes - len(df_FASE_final)} | quedan: {len(df_FASE_final)}")

Ha descartar imputaciones sin fase del ERP: 1000 | quedan: 16294


## 6. Fases sin tiempo teórico

Las fases sin tiempo en `PreparationTime` ni en `ExecutionTime` no son útiles: sin tiempo teórico no hay ratio.
Cuando se creó la OF, para esas fases no se estimó (o no se conocía) el tiempo teórico y quedó en 0.

In [71]:
teorico_cero = df_FASE_final[(df_FASE_final["PreparationTime"] == 0) & (df_FASE_final["ExecutionTime"] == 0)]
print(f"Fases con tiempo teórico 0: {len(teorico_cero)}\n")
print("Tiempo real bruto de esas fases (minutos):")
print(teorico_cero["MinutosReal_total"].describe().to_string(), "\n")
print("Por grupo de máquina:")
print(teorico_cero["NombreGrupo"].value_counts().to_string())

Fases con tiempo teórico 0: 1370

Tiempo real bruto de esas fases (minutos):
count    1370.000000
mean      162.499088
std       130.332980
min         0.166670
25%        65.225000
50%       128.266665
75%       231.137500
max       860.850000 

Por grupo de máquina:
NombreGrupo
Centre de Mecanitzat 5 Eixos petites        429
Torns Control Numèric                       360
Centres de Mecanitzat Vertical COURES       269
Centres de Mecanitzat Vertical ALUMINIS     137
Centres de Mecanitzat Horitzontal COURES    121
Premses Conformats                           23
Torns Puma (Taller COURES)                   19
Màquines Secció Flexibles                     8
Màquines de Taladrat Profund                  4


> Estas fases tienen imputaciones correctas, pero se eliminan porque el proyecto necesita el tiempo teórico, y en el ERP no se han especificado.

In [72]:
antes = len(df_FASE_final)
df_FASE_final = df_FASE_final[(df_FASE_final["PreparationTime"] != 0) | (df_FASE_final["ExecutionTime"] != 0)]
embudo["Fases con tiempos teóricos"] = len(df_FASE_final)
print(f"Descartadas (tiempo teórico 0): {antes - len(df_FASE_final)} | quedan: {len(df_FASE_final)}")

Descartadas (tiempo teórico 0): 1370 | quedan: 14924


## 7. Tipos de fase fuera de la población de estudio

Se revisa cuántos tipos de fase hay entre las fases de las máquinas productivas.

In [73]:
print(f"Tipos de fase según su código: {df_FASE_final['smFaseNme'].nunique()}")
print(f"Tipos de fase según su nombre: {df_FASE_final['Description'].nunique()}")

Tipos de fase según su código: 50
Tipos de fase según su nombre: 149


Con oficina técnica se acuerda un listado de tipos de fase a descartar: aunque se imputan en máquinas productivas,
corresponden a condiciones fuera de la población de estudio.

In [74]:
tipos_no_productivos = ["MAR", "PM", "AKS", "AKS-ALU", "ASO", "TRT", "TRT1",
                        "OFI", "OFI-ALU", "OFI-TCN", "SOL", "TCN-EXT"]

antes = len(df_FASE_final)
df_FASE_final = df_FASE_final[~df_FASE_final["smFaseNme"].isin(tipos_no_productivos)]
embudo["Fases de trabajos válidos"] = len(df_FASE_final)
print(f"Eliminadas por tipo de fase: {antes - len(df_FASE_final)} | quedan: {len(df_FASE_final)}")

embudo["OF finales"] = df_FASE_final["smOrdenId"].nunique()
print(f"OF a las que pertenecen: {embudo['OF finales']}")

Eliminadas por tipo de fase: 235 | quedan: 14689
OF a las que pertenecen: 6501


## 8. Guardado

In [75]:
df_FASE_final.to_parquet(PROCESSED / "df_FASE_final.parquet")
guardar_embudo(embudo, "05")

## Resumen

Evolución de las fases del ERP y de su unión con las del MES (el embudo completo, con las dos ramas, está en el
notebook 07):

In [76]:
tabla_embudo(embudo, "Número")

,Número
Fases iniciales,116055
Fases sin externas,107542
Fases en OF válidas,69986
Fases con imputaciones validadas,16294
Fases con tiempos teóricos,14924
Fases de trabajos válidos,14689
OF finales,6501
